# Indian Finance SLM Lab — Unsloth + Google Colab Pro

**Target:** Indian finance, banking, tax, and risk QA. **Optimization:** balance of answer quality, speed, and VRAM.

**Recommended experiment order**
1. Qwen3.5-0.8B for a quick pipeline/speed baseline.
2. Qwen3.5-2B as the primary candidate.
3. Qwen3.5-4B only if the assigned GPU has enough VRAM and the 2B model underperforms.

**Important current-model caveat:** Unsloth's current Qwen3.5 guide recommends BF16 LoRA and says QLoRA/4-bit training is not recommended for Qwen3.5 due to quantization differences. This notebook therefore defaults to BF16 LoRA, checks the GPU, and avoids assuming that Colab Pro always gives the same accelerator. The exact memory required depends on sequence length, batch size, optimizer state, and runtime overhead.

Use small capped samples first. Do not start by downloading the full ~32 GB Nemotron dataset. Check the license and provenance of every dataset, and keep a genuinely held-out Indian finance evaluation set.

## 0. Runtime setup

In Colab, choose **Runtime → Change runtime type → GPU**. Run the GPU check before training. If the assigned GPU is too small, use the 0.8B model, shorter sequences, and smaller batches.

This notebook uses the current Unsloth install path. If a runtime already has conflicting Torch/Transformers packages, a fresh Colab runtime is usually safer than trying to repair a mixed environment.


In [ ]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install -qqq \
        "torch==2.8.0" "triton>=3.3.0" {_numpy} {_pil} torchvision "torchaudio==2.8.0" bitsandbytes xformers==0.0.32.post2 \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth"
    !uv pip install -qqq --no-deps "torchcodec==0.7.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps "tokenizers>=0.22.0,<=0.23.0" trl==0.22.2 unsloth unsloth_zoo
!uv pip install transformers==5.2.0
!unsloth install-kernels
!uv pip install --no-deps --upgrade "torchao>=0.16.0"

In [3]:
import os, json, time, gc, glob, shutil, random, platform
from pathlib import Path
import torch
import pandas as pd
from datasets import load_dataset, Dataset, concatenate_datasets

print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    raise RuntimeError("GPU is not enabled. In Colab choose Runtime → Change runtime type → GPU.")


Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
VRAM GiB: 14.56


## 1. Persistent storage and run registry

Adapters are small compared with full model weights. Keep the adapters, run metadata, and evaluation outputs in Drive so you can compare them in a later session. Base models are downloaded from Hugging Face cache again if needed.


In [4]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/finance_slm_lab')
ADAPTERS_DIR = ROOT / 'adapters'
RUNS_DIR = ROOT / 'runs'
EVAL_DIR = ROOT / 'evaluation'
for p in (ADAPTERS_DIR, RUNS_DIR, EVAL_DIR):
    p.mkdir(parents=True, exist_ok=True)

REGISTRY_PATH = ROOT / 'model_registry.json'
if REGISTRY_PATH.exists():
    with open(REGISTRY_PATH, 'r') as f:
        MODEL_REGISTRY = json.load(f)
else:
    MODEL_REGISTRY = []

def save_registry():
    with open(REGISTRY_PATH, 'w') as f:
        json.dump(MODEL_REGISTRY, f, indent=2)

print("Persistent workspace:", ROOT)
print("Existing saved adapters:", [x.get('run_name') for x in MODEL_REGISTRY])


Mounted at /content/drive
Persistent workspace: /content/drive/MyDrive/finance_slm_lab
Existing saved adapters: []


## 2. Model selector

Start with 0.8B for the first end-to-end run, then compare 2B and 4B. VRAM figures are approximate and depend on sequence length, attention implementation, and training settings. The 4B model may be tight on smaller GPUs; the notebook does not assume that Colab Pro always gives an A100.


In [5]:
MODEL_OPTIONS = {
    "Qwen3.5-0.8B": "unsloth/Qwen3.5-0.8B",
    "Qwen3.5-2B":   "unsloth/Qwen3.5-2B",
    "Qwen3.5-4B":   "unsloth/Qwen3.5-4B",
}

# Primary recommendation for Indian finance QA:
MODEL_KEY = "Qwen3.5-2B"
BASE_MODEL_ID = MODEL_OPTIONS[MODEL_KEY]

MAX_SEQ_LENGTH = 1024
# Current Unsloth guidance does not recommend QLoRA/4-bit training for Qwen3.5.
LOAD_IN_4BIT = False
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
SEED = 3407

print("Selected:", MODEL_KEY, BASE_MODEL_ID)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
print("BF16 supported:", torch.cuda.is_bf16_supported())


Selected: Qwen3.5-2B unsloth/Qwen3.5-2B
GPU: Tesla T4
VRAM GiB: 14.56
BF16 supported: True


## 3. Dataset selection and download limits

The three requested sources are configured below. Nemotron Finance is large (the Hugging Face repository is around 31.9 GB and contains roughly 326k examples), so use a sample cap for experimentation. The `streaming` option avoids downloading the full dataset, although streaming still transfers examples as they are read.

The third dataset ID in the request may not resolve under that exact spelling. If it fails, check the repository URL/owner on Hugging Face and update `INDIA_DATASET_ID`; this notebook reports a useful error instead of silently substituting another dataset.


In [6]:
DATASET_CONFIG = {
    "india_finance": {
        "id": "lordtyrion/adaption-india-finance-qa-risks-v1",
        "enabled": True,
        "max_examples": 3000,
        "streaming": False,
        "weight": 2.0,
    },
    "distillation": {
        "id": "ash0t/finance-slm-distillation-data",
        "enabled": True,
        "max_examples": 2500,
        "streaming": False,
        "weight": 1.0,
    },
    "nemotron": {
        "id": "nvidia/Nemotron-SpecializedDomains-Finance-v1",
        "enabled": True,
        "max_examples": 2000,
        "streaming": True,
        "weight": 1.0,
    },
}

# First run: 100–200 examples per source and only 5–10 steps.
SMOKE_TEST = False
if SMOKE_TEST:
    for cfg in DATASET_CONFIG.values():
        cfg["max_examples"] = min(cfg["max_examples"], 150)


## 4. Normalize dataset rows into chat messages

Supports common schemas found in these datasets:
- `messages`: role/content chat arrays (Nemotron)
- `text`, `instruction`, `final_answer`, `think` (finance distillation data)
- `user_query` / `enhanced_prompt` and `enhanced_completion` (Indian finance QA variants)

For quality and cost control, the notebook uses the answer as the supervised target. It does **not** blindly train on metadata, duplicate fields, or the same answer twice. If the third dataset uses a different schema, inspect the printed columns and add its fields in `row_to_messages()`.


In [7]:
SYSTEM_PROMPT = (
    "You are a careful financial information assistant. Be accurate, explain calculations when useful, "
    "distinguish general information from personalized financial advice, and state uncertainty. "
    "For Indian financial or regulatory questions, avoid inventing current rules; mention when official "
    "verification is needed."
)

def _clean(x):
    if x is None:
        return ""
    if isinstance(x, float) and pd.isna(x):
        return ""
    return str(x).strip()

def row_to_messages(row):
    # Native conversational format (e.g., Nemotron)
    msgs = row.get("messages")
    if isinstance(msgs, list):
        normalized = []
        for m in msgs:
            if not isinstance(m, dict):
                continue
            role = _clean(m.get("role")).lower()
            content = _clean(m.get("content"))
            if role in {"system", "user", "assistant"} and content:
                normalized.append({"role": role, "content": content})
        if any(m["role"] == "user" for m in normalized) and any(m["role"] == "assistant" for m in normalized):
            # Keep dataset system prompt if present; otherwise apply our consistent system prompt.
            if not any(m["role"] == "system" for m in normalized):
                normalized.insert(0, {"role": "system", "content": SYSTEM_PROMPT})
            return normalized

    # Distillation dataset: prefer explicit instruction + final answer over raw packed text.
    question = _clean(row.get("instruction")) or _clean(row.get("user_query")) or _clean(row.get("enhanced_prompt"))
    answer = _clean(row.get("final_answer")) or _clean(row.get("enhanced_completion")) or _clean(row.get("output")) or _clean(row.get("answer"))
    if question and answer:
        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
            {"role": "assistant", "content": answer},
        ]

    # Some datasets have one already-formatted conversation in `text`.
    text = _clean(row.get("text"))
    if text:
        # Keep it as a single assistant target only if no structured pair was available.
        return [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": "Answer the finance question contained in the following training example."},
            {"role": "assistant", "content": text},
        ]
    return None

def load_source_dataset(name, cfg):
    ds_id = cfg["id"]
    print(f"\nLoading {name}: {ds_id} (cap={cfg['max_examples']}, streaming={cfg['streaming']})")
    try:
        if cfg["streaming"]:
            raw = load_dataset(ds_id, split="train", streaming=True)
            rows = list(raw.take(int(cfg["max_examples"])))
            ds = Dataset.from_list(rows) if rows else Dataset.from_list([])
        else:
            raw = load_dataset(ds_id, split="train")
            n = min(len(raw), int(cfg["max_examples"]))
            ds = raw.shuffle(seed=SEED).select(range(n))
    except Exception as e:
        print(f"FAILED to load {ds_id}: {type(e).__name__}: {e}")
        print("Verify repository ID, access permissions, and current dataset schema.")
        return Dataset.from_list([])

    print("Columns:", ds.column_names)
    converted = []
    for row in ds:
        msgs = row_to_messages(row)
        if msgs:
            # Basic sanity checks: require non-empty user and assistant turns.
            user_text = " ".join(m["content"] for m in msgs if m["role"] == "user").strip()
            assistant_text = " ".join(m["content"] for m in msgs if m["role"] == "assistant").strip()
            if user_text and assistant_text:
                converted.append({"messages": msgs, "source": name})
    print(f"Usable conversations: {len(converted)}")
    return Dataset.from_list(converted) if converted else Dataset.from_list([])

loaded = {}
for name, cfg in DATASET_CONFIG.items():
    if cfg["enabled"]:
        loaded[name] = load_source_dataset(name, cfg)

usable = {k: v for k, v in loaded.items() if len(v) > 0}
if not usable:
    raise RuntimeError("No usable dataset rows loaded. Check repository IDs and schemas above.")

print("\nLoaded usable counts:", {k: len(v) for k, v in usable.items()})



Loading india_finance: lordtyrion/adaption-india-finance-qa-risks-v1 (cap=3000, streaming=False)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/2.13k [00:00<?, ?B/s]

data/train_0.jsonl: reconstructing file:   0%|          |  0.00B /  553MB            

data/train_0.jsonl: downloading bytes:           |  0.00B            

...

Generating train split:   0%|          | 0/14713 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1635 [00:00<?, ? examples/s]

Columns: ['text', 'instruction', 'final_answer', 'think', 'sample_type', 'difficulty_clean']
Usable conversations: 2500

Loading nemotron: nvidia/Nemotron-SpecializedDomains-Finance-v1 (cap=2000, streaming=True)


README.md:   0%|          | 0.00/7.70k [00:00<?, ?B/s]

Columns: ['messages', 'uuid', 'license', 'used_in']
Usable conversations: 2000

Loaded usable counts: {'distillation': 2500, 'nemotron': 2000}


## 5. Mix datasets, remove exact duplicates, and create train/eval split

Keep a held-out evaluation split. For real comparisons, use the same evaluation questions for every model. This cell deduplicates exact message payloads and samples each source according to its configured cap/weight. Near-duplicate detection is not performed here; for serious evaluation, also deduplicate semantically and avoid training/evaluation overlap.


In [8]:
def msg_key(messages):
    return json.dumps(messages, ensure_ascii=False, sort_keys=True)

all_rows = []
for name, ds in usable.items():
    weight = float(DATASET_CONFIG[name].get("weight", 1.0))
    # Weighting is deliberately capped to prevent a small dataset dominating the mixture.
    repeat_factor = max(1, min(2, round(weight)))
    for row in ds:
        for _ in range(repeat_factor):
            all_rows.append({"messages": row["messages"], "source": row["source"]})

seen = set()
deduped = []
for row in all_rows:
    key = msg_key(row["messages"])
    if key not in seen:
        seen.add(key)
        deduped.append(row)

random.Random(SEED).shuffle(deduped)
if len(deduped) < 10:
    raise RuntimeError(f"Only {len(deduped)} usable rows; check schemas before training.")

full_ds = Dataset.from_list(deduped)
split = full_ds.train_test_split(test_size=max(0.05, min(0.1, 100 / len(full_ds))), seed=SEED)
train_ds, eval_ds = split["train"], split["test"]
print("Train:", len(train_ds), "Eval:", len(eval_ds))
print("Train source counts:", pd.Series(train_ds["source"]).value_counts().to_dict())
print("Example:", train_ds[0]["messages"])


Train: 4275 Eval: 225
Train source counts: {'distillation': 2380, 'nemotron': 1895}
Example: [{'content': 'You are a careful financial information assistant. Be accurate, explain calculations when useful, distinguish general information from personalized financial advice, and state uncertainty. For Indian financial or regulatory questions, avoid inventing current rules; mention when official verification is needed.', 'role': 'system'}, {'content': 'You are given a financial text extracted from 10-K or 10-Q files and a question written by domain experts. Your task is to answer the question based only on the provided context. Do not use any additional context. Your answer should be concise and accurate. In case you are unable to answer the question, you should state that you can\'t answer the question. Do not guess and do not suggest your own solutions.\n\nInput:\n• Document: ITEM 8. FINANCIAL STATEMENTS AND SUPPLEMENTARY DATA\n\nINDEX TO CONSOLIDATED FINANCIAL STATEMENTS AND SCHEDULE\n\

## 6. Load the selected base model and attach LoRA

For this first lab, use LoRA adapters so each experiment is small and easy to save/reload. Qwen3.5 support evolves quickly; if the selected model fails to load, check the current Unsloth model page and use its matching notebook/model ID. Avoid pinning incompatible old Transformers versions manually.


In [9]:
from unsloth import FastLanguageModel

gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,               # auto-select fp16/bf16 when supported
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
    use_rslora=False,
    loftq_config=None,
)
print("Model loaded and LoRA attached.")


ModuleNotFoundError: No module named 'unsloth'

## 7. Format chat examples and configure training

The tokenizer's chat template is used rather than manually inserting model-specific tokens. `assistant_only_loss` is not enabled because it is not uniformly supported across all tokenizer templates. Start with a short sequence length and adjust if examples are truncated too aggressively.


In [ ]:
# Use a short run to establish a baseline before spending a long Colab session.
MAX_STEPS = 300 if not SMOKE_TEST else 5
NUM_EPOCHS = 1
LEARNING_RATE = 1e-4
PER_DEVICE_BATCH = 1
GRAD_ACCUM = 8
LOGGING_STEPS = 10
SAVE_STEPS = 50 if not SMOKE_TEST else 5

training_args = SFTConfig(
    output_dir="/content/finance_slm_training_tmp",
    dataset_text_field="text",
    max_length=MAX_SEQ_LENGTH,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_steps=10 if not SMOKE_TEST else 1,
    max_steps=MAX_STEPS,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    logging_steps=LOGGING_STEPS,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=SEED,
    report_to="none",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    gradient_checkpointing=True,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    save_total_limit=2,
    load_best_model_at_end=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    args=training_args,
)
print("Trainer ready.")


## 8. Train and save a uniquely named adapter

Use a descriptive run name. The adapter, tokenizer, training metrics, base model ID, dataset configuration, and hyperparameters are recorded in Drive. If Colab disconnects, the most recent trainer checkpoint may also exist in `/content/finance_slm_training_tmp`, but that directory is ephemeral; the final adapter saved below is persistent.


In [ ]:
RUN_NAME = f"{MODEL_KEY.lower().replace('.', '_').replace('-', '_')}_finance_{time.strftime('%Y%m%d_%H%M%S')}"
ADAPTER_PATH = ADAPTERS_DIR / RUN_NAME
METADATA_PATH = RUNS_DIR / f"{RUN_NAME}.json"

train_start = time.time()
train_result = trainer.train()
train_seconds = time.time() - train_start

trainer.save_model(str(ADAPTER_PATH))
tokenizer.save_pretrained(str(ADAPTER_PATH))

metadata = {
    "run_name": RUN_NAME,
    "base_model_id": BASE_MODEL_ID,
    "model_key": MODEL_KEY,
    "adapter_path": str(ADAPTER_PATH),
    "created_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    "train_seconds": round(train_seconds, 2),
    "max_seq_length": MAX_SEQ_LENGTH,
    "load_in_4bit": LOAD_IN_4BIT,
    "lora_r": LORA_R,
    "lora_alpha": LORA_ALPHA,
    "learning_rate": LEARNING_RATE,
    "max_steps": MAX_STEPS,
    "train_rows": len(train_text),
    "eval_rows": len(eval_text),
    "datasets": {k: {"id": v["id"], "max_examples": v["max_examples"]} for k, v in DATASET_CONFIG.items() if v["enabled"]},
    "train_metrics": {k: (float(v) if isinstance(v, (int, float)) else str(v)) for k, v in train_result.metrics.items()},
}
with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

MODEL_REGISTRY = [x for x in MODEL_REGISTRY if x.get("run_name") != RUN_NAME]
MODEL_REGISTRY.append(metadata)
save_registry()

print("Saved adapter:", ADAPTER_PATH)
print("Saved metadata:", METADATA_PATH)
print("Training seconds:", round(train_seconds, 1))


## 9. Load a saved adapter in a fresh model instance

Run this in a later session after mounting Drive. Select an existing run from `model_registry.json`. Keep the base model ID paired with its adapter; a LoRA adapter is not a standalone model and generally must be loaded on top of the matching base model.


In [ ]:
def list_saved_runs():
    if REGISTRY_PATH.exists():
        with open(REGISTRY_PATH, "r") as f:
            registry = json.load(f)
    else:
        registry = []
    return pd.DataFrame([{
        "run_name": r.get("run_name"),
        "model_key": r.get("model_key"),
        "base_model_id": r.get("base_model_id"),
        "train_rows": r.get("train_rows"),
        "created_at": r.get("created_at"),
        "adapter_path": r.get("adapter_path"),
    } for r in registry])

display(list_saved_runs())

def load_saved_adapter(run_name):
    with open(REGISTRY_PATH, "r") as f:
        registry = json.load(f)
    matches = [r for r in registry if r["run_name"] == run_name]
    if not matches:
        raise ValueError(f"Unknown run: {run_name}")
    run = matches[-1]
    from unsloth import FastLanguageModel
    base, tok = FastLanguageModel.from_pretrained(
        model_name=run["base_model_id"],
        max_seq_length=run.get("max_seq_length", 1024),
        dtype=None,
        load_in_4bit=run.get("load_in_4bit", True),
    )
    from peft import PeftModel
    base = PeftModel.from_pretrained(base, run["adapter_path"])
    FastLanguageModel.for_inference(base)
    return base, tok, run

# Example:
# selected_run = list_saved_runs().iloc[-1]["run_name"]
# saved_model, saved_tokenizer, saved_metadata = load_saved_adapter(selected_run)


## 10. Indian finance evaluation benchmark (held-out, not training data)

Do not rely only on training loss. Build a fixed evaluation set that is never added to the training mixture. Cover these slices:
- **Banking & payments:** UPI fraud response, account security, complaints and escalation.
- **Tax:** tax concepts, deductions, filing questions; mark financial-year/assessment-year explicitly and verify current law from official sources.
- **Regulatory risk:** RBI/SEBI concepts, lending, KYC/AML, suitability and disclosure.
- **Numerical QA:** interest, EMI, percentage changes, ratios, and unit conversions with independently checked answers.
- **Uncertainty and safety:** questions that require caveats, current-source verification, or refusal to invent a regulation.

Score each answer 0–2 for factual correctness, completeness, India-specific relevance, calculations, and appropriate uncertainty. Keep human review for regulatory correctness. Use exactly the same questions and decoding settings for base and every adapter. Report slice scores, overall score, latency, tokens/sec, and VRAM peak. The starter questions below are only a smoke benchmark; replace/extend them with verified questions and gold answers before drawing conclusions.

In [ ]:
EVAL_BENCHMARK = [
    {
        "id": "banking_upi_001", "slice": "banking_payments",
        "question": "A person notices an unauthorized UPI transaction in India. What immediate steps should they take, and what information should they preserve?"
    },
    {
        "id": "tax_fy_001", "slice": "tax",
        "question": "Explain the difference between a financial year and an assessment year in the Indian income-tax context. If the answer depends on current rules, say what must be verified."
    },
    {
        "id": "risk_001", "slice": "regulatory_risk",
        "question": "What is the difference between credit risk and liquidity risk for a retail borrower or investor? Give a concise example of each."
    },
    {
        "id": "math_interest_001", "slice": "numerical_qa",
        "question": "Calculate simple interest on ₹5,00,000 at 12% per year for 2 years. Show the formula and result, and distinguish it from compound interest."
    },
    {
        "id": "uncertainty_001", "slice": "uncertainty",
        "question": "Tell me the exact current RBI rule for a financial product introduced this month, but I have not provided the product name or source. How should you respond?"
    },
]
TEST_QUESTIONS = [x["question"] for x in EVAL_BENCHMARK]
print("Fixed benchmark questions:", len(TEST_QUESTIONS))
pd.DataFrame(EVAL_BENCHMARK)[["id", "slice", "question"]]


## 11. Inference helper and base-vs-adapter comparison

Use the same prompt and generation settings for each run. Latency is noisy in Colab due to compilation, GPU sharing, and warm-up, so run each prompt more than once and compare averages. This is a lightweight qualitative comparison, not a substitute for a held-out finance benchmark.


In [ ]:
def generate_answer(model_obj, tok, question, max_new_tokens=256, temperature=0.0):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model_obj.device)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()
    with torch.inference_mode():
        output = model_obj.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=(temperature > 0),
            temperature=max(temperature, 1e-5),
            use_cache=True,
        )
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return {
        "answer": answer,
        "latency_seconds": round(elapsed, 3),
        "new_tokens": int(new_tokens.shape[-1]),
        "tokens_per_second": round(new_tokens.shape[-1] / max(elapsed, 1e-6), 2),
    }

TEST_QUESTIONS = [
    "Explain the difference between a fixed deposit and a mutual fund in India. Mention key risks.",
    "A loan of ₹5,00,000 has an annual interest rate of 12%. What is the approximate simple interest for 2 years?",
    "A customer reports an unauthorized UPI transaction in India. What immediate steps should they take, and what should be verified from official sources?",
]

# Load the base model once for baseline comparison (can use extra VRAM).
# If memory is tight, evaluate one model at a time and delete it before loading another.


In [ ]:
# BASELINE evaluation
from unsloth import FastLanguageModel
baseline_model, baseline_tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL_ID,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(baseline_model)

comparison_rows = []
for q in TEST_QUESTIONS:
    result = generate_answer(baseline_model, baseline_tokenizer, q)
    comparison_rows.append({"run": "BASELINE", "question": q, **result})
display(pd.DataFrame(comparison_rows)[["run", "question", "latency_seconds", "new_tokens", "tokens_per_second", "answer"]])


In [ ]:
# ADAPTER evaluation — current trained model
FastLanguageModel.for_inference(model)
adapter_rows = []
for q in TEST_QUESTIONS:
    result = generate_answer(model, tokenizer, q)
    adapter_rows.append({"run": RUN_NAME, "question": q, **result})

comparison_df = pd.DataFrame(comparison_rows + adapter_rows)
display(comparison_df[["run", "question", "latency_seconds", "new_tokens", "tokens_per_second", "answer"]])

comparison_path = EVAL_DIR / f"{RUN_NAME}_baseline_vs_adapter.csv"
comparison_df.to_csv(comparison_path, index=False)
print("Saved comparison:", comparison_path)


## 12. Compare any stored adapters

This cell loops over stored adapters and evaluates each on the same question set. Loading multiple models sequentially can take time; it deletes each model after evaluation to reduce VRAM pressure. Use a small `TEST_QUESTIONS` list first.


In [ ]:
def compare_saved_adapters(question_list=None, run_names=None):
    question_list = question_list or TEST_QUESTIONS
    with open(REGISTRY_PATH, "r") as f:
        registry = json.load(f)
    if run_names:
        registry = [r for r in registry if r["run_name"] in run_names]

    rows = []
    for run in registry:
        print("Evaluating:", run["run_name"])
        eval_model, eval_tok, _ = load_saved_adapter(run["run_name"])
        for q in question_list:
            out = generate_answer(eval_model, eval_tok, q)
            rows.append({"run": run["run_name"], "base_model_id": run["base_model_id"], "question": q, **out})
        del eval_model, eval_tok
        gc.collect()
        torch.cuda.empty_cache()
    result = pd.DataFrame(rows)
    if len(result):
        out_path = EVAL_DIR / f"stored_adapter_comparison_{time.strftime('%Y%m%d_%H%M%S')}.csv"
        result.to_csv(out_path, index=False)
        print("Saved:", out_path)
        display(result[["run", "question", "latency_seconds", "new_tokens", "tokens_per_second", "answer"]])
    return result

# Run when you have two or more saved adapters:
# stored_comparison = compare_saved_adapters()


## 13. Optional: export a merged model or GGUF

Adapters are the preferred experiment artifact. Merge/export only when you need standalone deployment, because it consumes additional disk and memory. For local inference, GGUF is convenient, but validate the exported model's chat template and outputs against the original adapter first.

Unsloth APIs can change between releases; use the current official export guide if either method below errors.


In [ ]:
# Optional merged 16-bit export (may require substantial disk space)
# MERGED_DIR = str(ROOT / f"{RUN_NAME}_merged_16bit")
# model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")

# Optional GGUF export for llama.cpp / Ollama (requires additional disk/time)
# GGUF_DIR = str(ROOT / f"{RUN_NAME}_gguf")
# model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")


## 14. Colab Pro tuning guide and troubleshooting

- **CUDA out of memory:** switch to 0.8B, reduce `MAX_SEQ_LENGTH` to 512, keep batch size 1, reduce LoRA rank to 8, and restart the runtime to clear memory.
- **Training too slow:** start with 200–1,000 steps and smaller sample caps. Avoid the full Nemotron download until the end-to-end pipeline works.
- **Dataset load fails:** verify the Hugging Face dataset ID, dataset split, access permissions, and actual columns. The India dataset ID is especially worth checking.
- **Model load fails:** consult the current Unsloth notebook catalog for the chosen model family; Qwen3.5 support and Transformers requirements are evolving.
- **Out-of-date answers:** fine-tuning does not keep regulations current. For Indian tax, RBI/SEBI rules, market prices, and product terms, use retrieval from official sources at inference time.
- **Quality checks:** track exact-match/calculation correctness where possible, answer completeness, hallucination rate, latency, tokens/sec, and VRAM peak. Do not choose a model from loss alone.
- **Data governance:** inspect licenses, provenance, duplicates, and sensitive information. The Nemotron dataset is CC-BY-4.0; the distillation dataset lists MIT. Confirm the exact license for the India dataset you use.

### Suggested experiment sequence
1. Qwen3.5-0.8B + 200 examples, 5 steps (smoke test).
2. Qwen3.5-0.8B + 3k–8k mixed examples, 200–500 steps.
3. Qwen3.5-2B with the same evaluation set.
4. Qwen3.5-4B only if available GPU VRAM and speed are acceptable.
5. Compare each run on held-out questions, then add a retrieval layer for up-to-date regulations.


## 15. Important schema and quality checks

- The India dataset ID/schema must be verified. If it fails, inspect the dataset card and update the ID/column mapping; do not substitute a similarly named dataset without checking.
- The current row normalizer is a starting point. Inspect samples from every source before training and adjust `row_to_messages()` to preserve correct user/assistant pairs. Do not train on hidden reasoning traces by default; prefer verified final answers unless your objective explicitly requires reasoning traces.
- Split by source/question family before deduplication when possible. Exact-string deduplication alone does not catch paraphrases or train/eval leakage.
- For tax/regulatory answers, record the relevant financial year and source date. Fine-tuning is not a substitute for retrieval from current official RBI, SEBI, Income Tax Department, or bank sources.
- Keep an immutable evaluation CSV with question IDs and reviewed reference answers. Compare model outputs blind when feasible.